In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
data = spark.table("electronics_retailer_clg.bronze.sales")

In [0]:
from pyspark.sql.functions import col, count,when

null_counts = data.select([count(when(col(c).isNull(),c)).alias(c) for c in data.columns])
display(null_counts)

In [0]:

import pyspark.sql.functions as F

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("order_date", data)
data = standardize_date("delivery_date", data)
display(data)

In [0]:

def dataTypeCast(df,column,dataType):
    return df.withColumn(column,F.trim(F.col(column)).cast(dataType))
data = dataTypeCast(data,'quantity','int')
data = dataTypeCast(data,'storekey','int')
data = dataTypeCast(data,'productkey','int')
data = dataTypeCast(data,'customerkey','int')
data = dataTypeCast(data,'line_item','int')
data = dataTypeCast(data,'currency_code','string')
data = dataTypeCast(data,'order_number','int')
display(data)

In [0]:
data.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("electronics_retailer_clg.silver.sales")

print("Sales cleaned perfectly for your dataset")

In [0]:
# from pyspark.sql.functions import col, trim, when, expr

# df = spark.table("electronics_retailer_clg.bronze.sales")

# df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

# for c in df.columns:
#     df = df.withColumn(c, trim(col(c)))

# df = df.withColumn(
#     "order_date",
#     expr("""
#         coalesce(
#             try_to_date(order_date, 'M/d/yyyy'),
#             try_to_date(order_date, 'M-d-yyyy'),
#             try_to_date(order_date, 'MM-dd-yyyy'),
#             try_to_date(order_date, 'yyyy-MM-dd'),
#             try_to_date(order_date, 'MM/dd/yyyy'),
#             try_to_date(order_date, 'M/dd/yyyy'),
#             try_to_date(order_date, 'MM/d/yyyy')
#         )
#     """)
# )

# df = df.withColumn(
#     "delivery_date",
#     expr("""
#         coalesce(
#             try_to_date(delivery_date, 'M/d/yyyy'),
#             try_to_date(delivery_date, 'M-d-yyyy'),
#             try_to_date(delivery_date, 'MM-dd-yyyy'),
#             try_to_date(delivery_date, 'yyyy-MM-dd'),
#             try_to_date(delivery_date, 'MM/dd/yyyy'),
#             try_to_date(delivery_date, 'M/dd/yyyy'),
#             try_to_date(delivery_date, 'MM/d/yyyy')
#         )
#     """)
# )


# df = df.withColumn("order_number", col("order_number").cast("int")) \
#        .withColumn("customerkey", col("customerkey").cast("int")) \
#        .withColumn("storekey", col("storekey").cast("int")) \
#        .withColumn("productkey", col("productkey").cast("int")) \
#        .withColumn("quantity", col("quantity").cast("int"))


# # df = df.dropna(subset=["order_number", "quantity", "order_date"])

# # REMOVE INVALID DATA
# df = df.filter(col("quantity") > 0)

# # Fix invalid delivery dates (if delivery is before order, set to null)
# df = df.withColumn(
#     "delivery_date",
#     when(col("delivery_date") < col("order_date"), None)
#     .otherwise(col("delivery_date"))
# )

# df = df.withColumn("currency_code", trim(col("currency_code")))


# df = df.dropDuplicates(["order_number", "productkey"])

# # KEEP ONLY REQUIRED COLUMNS 
# df = df.select(
#     "order_number",
#     "order_date",
#     "delivery_date",
#     "customerkey",
#     "storekey",
#     "productkey",
#     "quantity",
#     "currency_code"
# )

# display(df)
# df.printSchema()


# df.write.format("delta") \
#     .mode("overwrite") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("electronics_retailer_clg.silver.sales")

# print("Sales cleaned perfectly for your dataset")